In [3]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU đang dùng: {torch.cuda.get_device_name(0)}")

BATCH_SIZE = 128
EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-5
ETA = 0.1

Thiết bị: cuda
GPU đang dùng: Tesla T4


In [4]:
!mkdir -p data
!wget -O ./data/cifar-10-python.tar.gz https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz

--2026-10-14 03:27:38--  https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz
Resolving data.brainchip.com (data.brainchip.com)... 146.59.209.152, 2001:41d0:301::31
Connecting to data.brainchip.com (data.brainchip.com)|146.59.209.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 170498071 (163M) [application/x-gzip]
Saving to: ‘./data/cifar-10-python.tar.gz’

./data/cifar-10-pyt 100%[===================>] 162.60M  5.92MB/s    in 22s     

2026-10-14 03:28:01 (7.34 MB/s) - ‘./data/cifar-10-python.tar.gz’ saved [170498071/170498071]



In [7]:
!tar -xzf ./data/cifar-10-python.tar.gz -C ./data/

In [8]:
import numpy as np
from torch.utils.data import Subset

# Khi dùng backbone pretrained (AlexNet), bắt buộc resize ảnh về 224x224 theo ImageNet chuẩn
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform)
train_eval_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
test_eval_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=eval_transform)

y_train = np.array(train_raw.targets)
y_test = np.array(test_eval_raw.targets)

# Cố định Seed ngẫu nhiên để chia tập đồng bộ với nhóm
SEED = 2026
rng = np.random.default_rng(SEED)

# 1. Tập Train: 500 ảnh mỗi lớp -> 5.000 ảnh
train_idx = []
for c in range(10):
    idx_c = np.where(y_train == c)[0]
    train_idx.extend(rng.choice(idx_c, size=500, replace=False).tolist())
train_idx = np.array(sorted(train_idx), dtype=np.int64)

# 2. Tập Query: 100 ảnh mỗi lớp từ tập test -> 1.000 ảnh
query_idx = []
for c in range(10):
    idx_c = np.where(y_test == c)[0]
    query_idx.extend(rng.choice(idx_c, size=100, replace=False).tolist())
query_idx = np.array(sorted(query_idx), dtype=np.int64)

# 3. Tập Retrieval Gallery cho mAP@All: 59.000 ảnh
train_all_idx = np.arange(len(y_train))
test_remain_idx = np.array([i for i in range(len(y_test)) if i not in set(query_idx.tolist())], dtype=np.int64)

train_dataset = Subset(train_raw, train_idx)
query_dataset = Subset(test_eval_raw, query_idx)
db_train_part = Subset(train_eval_raw, train_all_idx)
db_test_part = Subset(test_eval_raw, test_remain_idx)
database_dataset = torch.utils.data.ConcatDataset([db_train_part, db_test_part])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
query_loader = DataLoader(query_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
database_loader = DataLoader(database_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training: {len(train_dataset)}")
print(f"Query: {len(query_dataset)}")
print(f"mAP@All gallery: {len(database_dataset)}")

Training: 5000
Query: 1000
mAP@All gallery: 59000


In [9]:
import torch.nn.functional as F
import torchvision.models as models

class DPSHNet(nn.Module):
    def __init__(self, bit_size):
        super(DPSHNet, self).__init__()
        weights = models.AlexNet_Weights.DEFAULT
        original_model = models.alexnet(weights=weights)
        self.features = original_model.features
        self.avgpool = original_model.avgpool

        # Tận dụng nguyên vẹn trọng số ImageNet của 2 tầng fc6 và fc7
        self.fc6 = original_model.classifier[1]
        self.relu6 = original_model.classifier[2]
        self.drop6 = original_model.classifier[0]

        self.fc7 = original_model.classifier[4]
        self.relu7 = original_model.classifier[5]
        self.drop7 = original_model.classifier[3]

        # Chỉ khởi tạo ngẫu nhiên duy nhất tầng ánh xạ ra bit băm
        self.hash_layer = nn.Linear(4096, bit_size)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.drop6(self.relu6(self.fc6(x)))
        x = self.drop7(self.relu7(self.fc7(x)))
        u = self.tanh(self.hash_layer(x))
        return u

class DPSHLoss(nn.Module):
    def __init__(self, eta=0.1):
        super(DPSHLoss, self).__init__()
        self.eta = eta

    def forward(self, u, y):
        s = (y.unsqueeze(1) == y.unsqueeze(0)).float()
        theta = 0.5 * torch.matmul(u, u.t())

        # Binary Cross Entropy with Logits chống tràn số tuyệt đối
        loss_likelihood = F.binary_cross_entropy_with_logits(theta, s)

        b = torch.sign(u)
        loss_quant = self.eta * torch.mean((b - u) ** 2)

        return loss_likelihood + loss_quant

print("Đã cập nhật cấu hình tối ưu giữ nguyên trọng số AlexNet Pretrained.")

Đã cập nhật cấu hình tối ưu giữ nguyên trọng số AlexNet Pretrained.


In [10]:
def extract_hash_codes(loader, model):
    """
    Trích xuất vector nhị phân b = sign(u) và nhãn tương ứng từ DataLoader
    """
    model.eval()
    all_codes, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            u = model(imgs)
            b = torch.sign(u)
            # Chuyển đổi mã về dạng {-1, 1} hoặc {0, 1} (dùng -1, 1 để tính Hamming nhanh)
            all_codes.append(b.cpu().numpy())
            all_labels.append(lbls.numpy())
    return np.vstack(all_codes), np.concatenate(all_labels)

def compute_map_all(query_codes, query_labels, db_codes, db_labels):
    """
    Tính Mean Average Precision trên toàn bộ cơ sở dữ liệu (mAP@All)
    Khớp với giao thức đánh giá của DPSH trong Table IV
    """
    num_query = query_labels.shape[0]
    total_ap = 0.0

    print(f"--> Đang tính toán mAP@All cho {num_query} ảnh truy vấn trên {db_codes.shape[0]} ảnh gallery...")
    for i in range(num_query):
        # Khoảng cách Hamming = số vị trí khác dấu giữa 2 vector
        hamming_dist = np.count_nonzero(query_codes[i] != db_codes, axis=1)
        sort_indices = np.argsort(hamming_dist)

        # Xác định các ảnh trong database có cùng lớp với query
        matched = (query_labels[i] == db_labels[sort_indices])
        total_relevant = np.sum(matched)

        if total_relevant == 0:
            continue

        cumsum = np.cumsum(matched)
        ranks = np.arange(1, len(matched) + 1)
        precision_at_k = cumsum / ranks

        ap = np.sum(precision_at_k * matched) / total_relevant
        total_ap += ap

    return total_ap / num_query

print("Đã nạp xong các hàm đánh giá mAP@All.")

Đã nạp xong các hàm đánh giá mAP@All.


In [11]:
# ==========================================
# CELL 6: HUẤN LUYỆN VÀ ĐỐI CHIẾU TABLE IV (24, 32, 48 BITS)
# ==========================================
target_bits = [24, 32, 48]
paper_reference = {24: 72.70, 32: 74.40, 48: 75.70}
results = {}

EPOCHS = 40  # Đủ số epoch để loss hội tụ sâu xuống đáy 0.2 - 0.3

print("BẮT ĐẦU CHUỖI THỰC NGHIỆM DPSH ĐỐI CHIẾU TABLE IV (24, 32, 48 BITS)")

for bit in target_bits:
    print("\n" + "=" * 58)
    print(f"--> HUẤN LUYỆN DPSH VỚI {bit} BITS ({EPOCHS} EPOCHS)")
    print("=" * 58)

    model = DPSHNet(bit_size=bit).to(DEVICE)
    criterion = DPSHLoss(eta=ETA)

    # Chiến lược học chuyển giao: giữ backbone ổn định, đẩy nhanh tầng băm
    optimizer = optim.Adam([
        {'params': model.features.parameters(), 'lr': 1e-5},
        {'params': model.fc6.parameters(), 'lr': 5e-5},
        {'params': model.fc7.parameters(), 'lr': 5e-5},
        {'params': model.hash_layer.parameters(), 'lr': 2e-4}
    ], weight_decay=WEIGHT_DECAY)

    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 32], gamma=0.2)

    start_time = time.time()
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            u = model(imgs)
            loss = criterion(u, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()
        avg_loss = running_loss / len(train_loader)
        if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch + 1) == EPOCHS:
            print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] - Loss: {avg_loss:.4f}")

    train_duration = (time.time() - start_time) / 60
    print(f"-> Thời gian huấn luyện xong {bit} bits: {train_duration:.2f} phút.")

    # Trích xuất mã băm trên Query (1.000 ảnh) và Retrieval Gallery (59.000 ảnh)
    print("-> Đang trích xuất mã băm nhị phân...")
    q_codes, q_labels = extract_hash_codes(query_loader, model)
    d_codes, d_labels = extract_hash_codes(database_loader, model)

    # Tính toán độ đo mAP@All
    map_score = compute_map_all(q_codes, q_labels, d_codes, d_labels) * 100
    results[bit] = map_score
    print(f"-> Kết quả thực nghiệm {bit} bits: {map_score:.2f}% | Tham chiếu Table IV: {paper_reference[bit]}%")

# ==========================================
# IN BẢNG TỔNG HỢP ĐỐI CHIẾU CHUẨN XÁC VỚI TABLE IV
# ==========================================
print("\n" + "=" * 62)
print(f"{'Mã Bit':<10} | {'Paper (Table IV, mAP@All)':<27} | {'Reproduced (Ours)':<20}")
print("-" * 62)
for bit in target_bits:
    print(f"{bit:<10} | {paper_reference[bit]:<27.2f}% | {results[bit]:<20.2f}%")
print("=" * 62)

BẮT ĐẦU CHUỖI THỰC NGHIỆM DPSH ĐỐI CHIẾU TABLE IV (24, 32, 48 BITS)

--> HUẤN LUYỆN DPSH VỚI 24 BITS (40 EPOCHS)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 187MB/s]


Epoch [01/40] - Loss: 0.7677
Epoch [10/40] - Loss: 0.5313
Epoch [20/40] - Loss: 0.3329
Epoch [30/40] - Loss: 0.2926
Epoch [40/40] - Loss: 0.2792
-> Thời gian huấn luyện xong 24 bits: 6.30 phút.
-> Đang trích xuất mã băm nhị phân...
--> Đang tính toán mAP@All cho 1000 ảnh truy vấn trên 59000 ảnh gallery...
-> Kết quả thực nghiệm 24 bits: 76.93% | Tham chiếu Table IV: 72.7%

--> HUẤN LUYỆN DPSH VỚI 32 BITS (40 EPOCHS)
Epoch [01/40] - Loss: 0.7893
Epoch [10/40] - Loss: 0.5375
Epoch [20/40] - Loss: 0.2947
Epoch [30/40] - Loss: 0.2366
Epoch [40/40] - Loss: 0.2249
-> Thời gian huấn luyện xong 32 bits: 6.73 phút.
-> Đang trích xuất mã băm nhị phân...
--> Đang tính toán mAP@All cho 1000 ảnh truy vấn trên 59000 ảnh gallery...
-> Kết quả thực nghiệm 32 bits: 77.12% | Tham chiếu Table IV: 74.4%

--> HUẤN LUYỆN DPSH VỚI 48 BITS (40 EPOCHS)
Epoch [01/40] - Loss: 0.8147
Epoch [10/40] - Loss: 0.5600
Epoch [20/40] - Loss: 0.2867
Epoch [30/40] - Loss: 0.2105
Epoch [40/40] - Loss: 0.1951
-> Thời gian hu